In [ ]:
## TOOK from CNN example code and A02 skelton as it had many useful parts, just edited somethjing to make it work
# CNN on CIFAR-10 — AlexNet (from scratch)
#trying to quickly do this as i said i lost all of my files before so may not look the cleanest
import torch, torch.nn as nn, torch.optim as optim
from torch.utils.data import DataLoader, random_split, Subset
from torchvision import datasets, transforms, models
import matplotlib.pyplot as plt, numpy as np
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import copy#for realy stoping




config = {#grabbed from a02 and editing as i go not all of config was used as this was from A02

    "prototyping": False,# True for Full data false for partial;
    "train_size_frac": 0.1, #10 percent as osaid

    "activation": "relu", # relu, leakyrelu, sigmoid
    "leakyrelu_negative_slope": 0.02,
    "dropout_probs": [0.2, 0.2, 0.0],
    "normalization": "layernorm", # "none", "batchnorm", "layernorm", "groupnorm" didnt use
    #
    "epochs": 15,#as said
    "learning_rate": 1e-3,#used
    "weight_decay": 1e-4,#
    "optimizer": "adam", #adam or sgd


    "lr_scheduler": None,# None, OneCycle, ReduceOnPlateau
    "OneCycle": {"max_lr": 0.01},


    "patience": 3,#early Stopping
}



device = torch.device('cuda:0') if torch.cuda.is_available() else torch.device('cpu')
print(device)#esnures gpu use A02

#1.Dataset and transforms
transform = transforms.Compose([
    transforms.Resize((64, 64)),#not sure about resize refer back to other code example 224 64
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5),
                         (0.5, 0.5, 0.5))
])


full_train   = datasets.CIFAR10("./data", train=True,  download=True, transform=transform)
test_dataset = datasets.CIFAR10("./data", train=False, download=True, transform=transform)
num_classes  = 10 #ao2 also had input_size here too

#Prototyping #greabbed from A02
PROTOTYPING = config["prototyping"]
train_size_frac = config["train_size_frac"]

if config["prototyping"]:
    # work with a fraction of the full dataset for quicker prototyping (e.g. 20%); keep
    # keep the train:val ratio = 4:1
    total = int(train_size_frac * len(full_train))
    train_size = int(0.7 * total)
    val_size   = int(0.15 * total)
    test_size  = total - train_size - val_size


    train_val_subset, _ = random_split(full_train,
                                       [total, len(full_train) - total])

    train_dataset, val_dataset, test_dataset= random_split(train_val_subset,
                                              [train_size, val_size,test_size])
else:
    #split in the 70 15 15 split
    train_size = int(0.7 * len(full_train))
    val_size   = int(0.15 * len(full_train))
    test_size  = len(full_train) - train_size - val_size
    train_dataset, val_dataset, test_dataset = random_split(full_train,
    [train_size, val_size, test_size]
)


train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True)
val_loader   = DataLoader(val_dataset,   batch_size=128, shuffle=False)
test_loader  = DataLoader(test_dataset,  batch_size=128, shuffle=False)



#2.Small CNN section just alexnet right now( i believe model 1 doesnt need anything else like layers yet hopefully)
#looked at using Alexnet and cnn example

#model = models.alexnet(weights=None)#no weigfhts
#model.classifier[6] = nn.Linear(model.classifier[6].in_features,10)
#model = model.to(device)

#criterion = nn.CrossEntropyLoss()
class SimpleCNN(nn.Module):
    def __init__(self, n_classes=10):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3,16,3,padding=1), nn.ReLU(), nn.MaxPool2d(2,2),
            nn.Conv2d(16,32,3,padding=1), nn.ReLU(), nn.MaxPool2d(2,2),
            nn.Conv2d(32,64,3,padding=1), nn.ReLU(), nn.MaxPool2d(2,2)
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64*8*8,256), nn.ReLU(),
            nn.Linear(256,n_classes)
        )
    def forward(self,x): return self.classifier(self.features(x))

model = SimpleCNN(n_classes=10).to(device)
criterion = nn.CrossEntropyLoss()

#ao2
if config["optimizer"] == "adam":# this may not be even needed tbh
    optimizer = optim.Adam(model.parameters(),
                           lr=config["learning_rate"],weight_decay=config ["weight_decay"])#A02 had more like decay,etc

else:
    optimizer = optim.SGD(model.parameters(),
                          lr=config["learning_rate"],weight_decay=config["weight_decay"], momentum=0.9)
# TODO 08: Add support for ReduceOnPlateau and OneCycle LRScheduler (separate) use configuration parameters instead of fixed arguments #A02
if not config["lr_scheduler"]:
        scheduler = None
elif config["lr_scheduler"] == "ReduceOnPlateau":
    #pass
    #scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, config["ReduceOnPlateau"])
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, **config.get("ReduceOnPlateau", {}))
elif config["lr_scheduler"] == "OneCycle":
    #pass
    scheduler = optim.lr_scheduler.OneCycleLR(
        optimizer,
        max_lr=config["OneCycle"]["max_lr"],
        epochs=config["epochs"],
        steps_per_epoch=len(train_loader)
    )
else:
    scheduler = None


#3. Train + validate

EPOCHS = config["epochs"]
train_losses, val_losses = [], []
train_accs,   val_accs   = [], []

patience = config["patience"]#this is for early stoping
best_val_loss = np.inf
patience_counter = 0
best_state = copy.deepcopy(model.state_dict())



for epoch in range(EPOCHS):
    model.train()
    total_loss, correct, total = 0, 0, 0
    for X, y in train_loader:
        X, y = X.to(device), y.to(device)
        optimizer.zero_grad()
        out = model(X);loss = criterion(out, y)
        loss.backward();optimizer.step()
        total_loss += loss.item() * X.size(0)

        if scheduler is not None and config["lr_scheduler"] == "OneCycle":
            scheduler.step()


        _, preds = torch.max(out, 1)
        correct += (preds == y).sum().item()
        total += y.size(0)

    train_loss = total_loss / total
    train_acc = correct / total

    # validation
    model.eval();vloss, vcorrect, vtotal = 0, 0, 0
    with torch.no_grad():
        for X, y in val_loader:
            X, y = X.to(device), y.to(device)
            out = model(X)
            loss = criterion(out, y)


            vloss += loss.item() * X.size(0)
            _, preds = torch.max(out, 1)
            vcorrect += (preds == y).sum().item()
            vtotal += y.size(0)

    val_loss = vloss / vtotal
    val_acc = vcorrect / vtotal
    if scheduler is not None and config["lr_scheduler"] == "ReduceOnPlateau":
        scheduler.step(val_loss)


    train_losses.append(train_loss)
    val_losses.append(val_loss)
    train_accs.append(train_acc)
    val_accs.append(val_acc)
    # --- Print progress ---#granned from A02
    print(f"Epoch {epoch+1}/{config['epochs']} | "
              f"Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f} | "
              f"Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.4f}")

#early stpoing below refer to a02
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        patience_counter = 0
        best_state = copy.deepcopy(model.state_dict())
    else:
        patience_counter += 1
        if patience_counter >= config["patience"]:
            print(f"Early stopping.")
            break
if best_state is not None:
    model.load_state_dict(best_state)

# 4.Evaluation like the plots and confusion matrix

fig, ax = plt.subplots(figsize=(8,5))

#curves
ax.plot(train_losses, label="Train Loss")
ax.plot(val_losses,  label="Val Loss")

#ax.plot(train_accs, label="Train Acc") #actually not needed for this only the loss aspect
#ax.plot(val_accs,   label="Val Acc")

ax.set_xlabel("Epoch")
ax.set_ylabel("Value")
ax.legend()
ax.grid(True)
ax.set_title(f"Epoch {epoch+1}/{config['epochs']} "
                     f"| Train Acc: {train_acc:.3f}, Val Acc: {val_acc:.3f}")

plt.show()


#just like in A02 with slighjt change to make it work
def evaluate(model, loader):
    model.eval()
    correct = 0
    total = 0
    all_preds, all_labels = [], []
    with torch.no_grad():
        for X, y in loader:
            X, y = X.to(device), y.to(device)
            outputs = model(X)

            _, preds = torch.max(outputs, 1)
            correct += (preds == y).sum().item()
            total += y.size(0)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(y.cpu().numpy())
    return correct / total, np.array(all_preds), np.array(all_labels)

test_acc, preds, labels = evaluate(model, test_loader)
print(f"Overall Test Accuracy: {test_acc:.4f}")

# Per-class accuracy
for i in range(num_classes):
    idx = labels == i
    acc_i = (preds[idx] == labels[idx]).mean()
    print(f"Class {i}: {acc_i:.4f}")

# Confusion Matrix
cm = confusion_matrix(labels, preds)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=range(num_classes))
disp.plot(cmap=plt.cm.Blues)
plt.show()